# UAE Property Finder Dataset: Dubai Real Estate Market Analysis

## Executive Summary
This notebook explores the **UAE Property Finder Comprehensive Dataset**. The UAE real estate market (particularly Dubai) is one of the most dynamic and hyper-competitive property markets in the world. 

We will analyze the residential and commercial markets, looking at pricing dynamics, community comparisons, and building a predictive model.

**Key Focus Areas:**
- Market category breakdowns (Rent vs Buy).
- High-value community hotspots.
- Predictive pricing baseline.


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")


## Section 2 - Data Loading
We load all 5 UAE sub-datasets.


In [ ]:
import os

base_path = '/kaggle/input/uae-property-finder/'
if not os.path.exists(base_path):
    base_path = 'UAE_Property_Finder_Kaggle/processed/'

datasets = {
    'Res_Buy': pd.read_csv(os.path.join(base_path, 'dubai_residential_buy.csv')),
    'Res_Rent': pd.read_csv(os.path.join(base_path, 'dubai_residential_rent.csv')),
    'Com_Buy': pd.read_csv(os.path.join(base_path, 'dubai_commercial_buy.csv')),
    'Com_Rent': pd.read_csv(os.path.join(base_path, 'dubai_commercial_rent.csv')),
    'New_Projects': pd.read_csv(os.path.join(base_path, 'dubai_new_projects.csv'))
}

for name, df in datasets.items():
    print(f"{name} Shape: {df.shape}")

display(datasets['Res_Rent'].head())


## Section 3 - Data Quality
Checking feature coverage across the datasets.


In [ ]:
# Null check for the largest dataset (Residential Rent)
nulls = datasets['Res_Rent'].isnull().sum()
print("Missing values in Residential Rent:\n", nulls[nulls > 0].sort_values(ascending=False))

# Duplicates
print("\nDuplicates in Residential Rent:", datasets['Res_Rent'].duplicated().sum())


## Section 4 - Market Overview
Comparing category sizes.


In [ ]:
sizes = {name: len(df) for name, df in datasets.items()}
fig = px.bar(x=list(sizes.keys()), y=list(sizes.values()), title='Number of Listings per Category', color=list(sizes.values()))
fig.update_layout(xaxis_title="Category", yaxis_title="Number of Listings")
fig.show()


**Interpretation:**
The rental market has significantly higher listing velocity and volume than the buy market in Dubai.


## Section 5 - Pricing Analysis
Analyzing the Residential Rent pricing distribution.


In [ ]:
df_rent = datasets['Res_Rent'].copy()

# Price Distribution
fig = px.histogram(df_rent[df_rent['price'] < df_rent['price'].quantile(0.99)], x='price', nbins=50, title='Residential Rent Price Distribution (Excl. Outliers)')
fig.show()

if 'price_per_sqft' in df_rent.columns:
    fig = px.histogram(df_rent[df_rent['price_per_sqft'] < df_rent['price_per_sqft'].quantile(0.99)], x='price_per_sqft', nbins=50, title='Price per SqFt Distribution')
    fig.show()


## Section 6 - Geographic Analysis
Identifying the most expensive communities for renting.


In [ ]:
# Clean location to get primary district
df_rent['district'] = df_rent['location'].apply(lambda x: str(x).split(',')[-2].strip() if len(str(x).split(',')) > 1 else str(x))

geo_price = df_rent.groupby('district').agg({'price': 'median', 'id': 'count'}).reset_index()
geo_price = geo_price[geo_price['id'] > 30].sort_values('price', ascending=False)

fig = px.bar(geo_price.head(15), x='district', y='price', title='Top 15 Most Expensive Districts (Median Rent)')
fig.show()


## Section 7 - Property Features
Analyzing structural configurations.


In [ ]:
if 'bedrooms' in df_rent.columns:
    bed_counts = df_rent['bedrooms'].value_counts().reset_index()
    bed_counts.columns = ['Bedrooms', 'Count']
    fig = px.pie(bed_counts, values='Count', names='Bedrooms', title='Proportion of Bedrooms in Rental Listings')
    fig.show()


## Section 8 - Correlation Analysis


In [ ]:
cols = ['price', 'bedrooms', 'bathrooms', 'size']
available_cols = [c for c in cols if c in df_rent.columns]
corr = df_rent[available_cols].corr()

fig = px.imshow(corr, text_auto=True, title='Feature Correlation Heatmap')
fig.show()


## Section 9 - Machine Learning Baseline
Building a predictive model for Dubai rent prices.


In [ ]:
ml_df = df_rent.dropna(subset=['price', 'bedrooms', 'bathrooms', 'size']).copy()

q_low = ml_df['price'].quantile(0.01)
q_hi  = ml_df['price'].quantile(0.99)
ml_df = ml_df[(ml_df['price'] > q_low) & (ml_df['price'] < q_hi)]

features = ['bedrooms', 'bathrooms', 'size']
X = ml_df[features]
y = ml_df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

preds = rf.predict(X_test)

print("--- Baseline Random Forest Metrics (Rental Prices) ---")
print(f"MAE:  {mean_absolute_error(y_test, preds):,.2f} AED")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, preds)):,.2f} AED")
print(f"R²:   {r2_score(y_test, preds):.4f}")

# Feature Importance
imp = pd.DataFrame({'Feature': features, 'Importance': rf.feature_importances_})
fig = px.bar(imp.sort_values('Importance', ascending=False), x='Feature', y='Importance', title='Random Forest Feature Importance')
fig.show()


## Section 10 - Market Insights
1. **Hyper-Velocity Rentals:** The rental market dwarfs the buy market, indicating high population transience and investor yield-generation.
2. **Size is King:** Property size strictly dominates pricing dynamics, vastly outweighing discrete bedroom counts in Dubai.
3. **Luxury Clusters:** Specific micro-communities (e.g., Palm Jumeirah, Downtown Dubai) drastically skew the upper echelons of the price distributions, suggesting severe inequality in district valuations.
